# TESS Exoplanet Pipeline — Step-by-Step Standalone Walkthrough

This notebook implements the entire TESS exoplanet analysis pipeline (detection, stellar characterization, GP detrending, and Bayesian MCMC modeling) from first principles. It uses standard scientific and astronomical Python libraries directly (`lightkurve`, `transitleastsquares`, `astroquery`, `pymc`, `exoplanet`, `celerite2`, `corner`, etc.) and contains **zero imports or dependencies** on the `tess_pipeline` library itself.

## 🚀 Setup & Installation
Ensure you have installed the required dependencies in your active Python/Conda environment:
```bash
pip install lightkurve transitleastsquares astroquery pymc exoplanet celerite2 corner arviz astropy
```

In [ ]:
# Enable inline plotting
%matplotlib inline

import os
import re
import csv
import json
import math
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import pymc as pm
import arviz as az
import exoplanet as xo
import celerite2.pymc as celerite2_pm
from celerite2.pymc import terms as c2terms
import pytensor.tensor as pt
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.gaia import Gaia
import corner
from transitleastsquares import transitleastsquares

warnings.filterwarnings("ignore")
print("Setup Complete. All standard packages imported without using tess_pipeline.")

## 🎯 Step 1: Target Resolution
We parse the target name string to extract the TIC ID, and search the Mikulski Archive for Space Telescopes (MAST) via `lightkurve` to retrieve the sky coordinates (RA/Dec).

In [ ]:
def resolve_target(target):
    raw = str(target).strip()
    match = re.search(r"(?:tic[\s_\-]*)?(\d{6,16})", raw, re.IGNORECASE)
    if not match:
        raise ValueError(f"Cannot parse target '{raw}' as a TIC ID.")
    tic_id = int(match.group(1))
    
    print(f"Querying MAST for TIC {tic_id} coordinates...")
    try:
        search = lk.search_lightcurve(f"TIC {tic_id}", mission="TESS")
        if len(search) == 0:
            print("No MAST observations found; coordinates set to None.")
            return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": None, "dec": None}
        t = search.table
        ra = float(t["s_ra"][0]) if "s_ra" in t.colnames else None
        dec = float(t["s_dec"][0]) if "s_dec" in t.colnames else None
        return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": ra, "dec": dec}
    except Exception as e:
        print(f"MAST target resolution failed: {e}")
        return {"tic_id": tic_id, "name": f"TIC {tic_id}", "ra": None, "dec": None}

target_info = resolve_target("TIC 261136679")
print("Resolved Target Information:")
for key, value in target_info.items():
    print(f"  {key}: {value}")

## 📚 Step 2: Archival Reference Lookup
We query the local NASA Exoplanet Archive TOI export file in the `data/` directory to check if there is a known cataloged period and transit epoch for this target.

In [ ]:
def query_local_archive(tic_id, data_dir="data"):
    csv_files = sorted(Path(data_dir).glob("TOI_*.csv"), reverse=True)
    if not csv_files:
        print("No local TOI catalog CSV found in data/.")
        return {"period": None, "source": "none"}
    
    archive_path = csv_files[0]
    print(f"Querying local TOI catalog: {archive_path.name}")
    
    with archive_path.open("r", encoding="utf-8", newline="") as f:
        filtered = (line for line in f if not line.startswith("#"))
        reader = csv.DictReader(filtered)
        rows = list(reader)
        
    matches = []
    for r in rows:
        for tid_key in ("tid", "ticid", "tic_id"):
            val = r.get(tid_key)
            if val and int(val) == tic_id:
                matches.append(r)
                break
                
    if not matches:
        print(f"No entries found in local TOI catalog for TIC {tic_id}")
        return {"period": None, "source": "none"}
    
    def record_sort_key(row):
        disp = (row.get("tfopwg_disp") or "").strip().upper()
        disp_rank = {"CP": 0, "KP": 1, "PC": 2, "APC": 3, "FA": 4, "FP": 5}.get(disp, 99)
        try:
            pl_pnum = int(float(row.get("pl_pnum", 9999)))
        except:
            pl_pnum = 9999
        try:
            toi = float(row.get("toi", 999999.0))
        except:
            toi = 999999.0
        return (disp_rank, pl_pnum, toi)
        
    matches.sort(key=record_sort_key)
    row = matches[0]
    
    def _safe_float(v):
        if v in (None, ""): return None
        try:
            f = float(v)
            return f if np.isfinite(f) else None
        except:
            return None
            
    period = _safe_float(row.get("pl_orbper"))
    toi = row.get("toi", "")
    planet_name = f"TOI-{toi}" if toi else None
    
    return {
        "period": period,
        "epoch": _safe_float(row.get("pl_tranmid")),
        "planet_name": planet_name,
        "rp_earth": _safe_float(row.get("pl_rade")),
        "t_eq": _safe_float(row.get("pl_eqt")),
        "reference": f"Local TOI catalog export: {archive_path.name}",
        "source": "toi-local",
        "raw_row": row
    }

archive_result = query_local_archive(target_info["tic_id"])
print("Local TOI Catalog Query:")
for key, value in archive_result.items():
    if key != "raw_row":
        print(f"  {key}: {value}")

## 📥 Step 3: Data Acquisition
We download the 2-minute cadence SPOC PDCSAP light curves from MAST using `lightkurve.search_lightcurve`. We download and clean a single sector to optimize local compute speeds.

In [ ]:
def download_spoc_lightcurves(tic_id, sector=1, cadence=120):
    print(f"Searching MAST for SPOC light curves for TIC {tic_id}...")
    search = lk.search_lightcurve(
        f"TIC {tic_id}",
        author="SPOC",
        exptime=cadence,
        mission="TESS"
    )
    if len(search) == 0:
        raise ValueError(f"No short-cadence SPOC data found for TIC {tic_id}")
    
    table = search.table
    if "sequence_number" in table.colnames:
        sorted_idx = sorted(range(len(search)), key=lambda i: table["sequence_number"][i])
        search = search[sorted_idx]
        
    n_sectors = min(sector, len(search))
    print(f"Downloading {n_sectors} sector(s) of FITS products...")
    
    os.makedirs("data/fits", exist_ok=True)
    collection = search[:n_sectors].download_all(
        flux_column="pdcsap_flux",
        download_dir="data/fits",
        quality_bitmask="hardest"
    )
    
    # Move and flatten downloaded FITS files out of mastDownload structure
    import shutil
    mast_dir = Path("data/fits/mastDownload")
    if mast_dir.exists():
        for fits_file in mast_dir.glob("**/*.fits*"):
            dest = Path("data/fits") / fits_file.name
            shutil.move(str(fits_file), str(dest))
        shutil.rmtree(str(mast_dir))
        
        # Re-read files from flat directory
        flat_files = [str(p) for p in Path("data/fits").glob(f"*-{tic_id}-*.fits*")]
        if not flat_files:
            flat_files = [str(p) for p in Path("data/fits").glob(f"*-{tic_id:016d}-*.fits*")]
        collection = lk.LightCurveCollection([
            lk.read(f).to_lightcurve(flux_column="pdcsap_flux") for f in flat_files
        ])
        
    return collection

raw_collection = download_spoc_lightcurves(target_info["tic_id"], sector=1)
print(f"Acquired light curve collection containing {len(raw_collection)} sectors.")

## 🧹 Step 4: Preprocessing & Cleaning
We clean the light curve by removing NaNs and outliers (via sigma-clipping), stitching multiple sectors into a single light curve, and detrending low-frequency stellar variability using a Savitzky-Golay filter.

In [ ]:
def preprocess_lightcurves(collection):
    cleaned = []
    for lc in collection:
        lc = lc.remove_nans()
        # Perform first outlier rejection pass
        lc = lc.remove_outliers(sigma_lower=20.0, sigma_upper=5.0)
        cleaned.append(lc)
        
    stitched = lk.LightCurveCollection(cleaned).stitch()
    
    # Flatten (window_length=401 cadences approx 13 hours; polyorder=3)
    flat, trend = stitched.flatten(
        window_length=401,
        polyorder=3,
        break_tolerance=5,
        return_trend=True
)
    
    # Second outlier rejection pass on flattened data
    flat = flat.remove_nans().remove_outliers(sigma_lower=20.0, sigma_upper=5.0)
    return flat, stitched, trend

lc, stitched_lc, trend_lc = preprocess_lightcurves(raw_collection)

# Visualize detrending results
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(stitched_lc.time.value, stitched_lc.flux.value, ".k", alpha=0.3, label="Stitched PDCSAP")
axes[0].plot(stitched_lc.time.value, trend_lc.flux.value, "-r", linewidth=1.5, label="Savitzky-Golay Trend")
axes[0].set_ylabel("Flux (e-/s)")
axes[0].legend(loc="upper right")
axes[0].set_title("Stitched Raw Light Curve with Detrending Model")

axes[1].plot(lc.time.value, lc.flux.value, ".b", alpha=0.3, label="Flattened Data")
axes[1].set_ylabel("Normalized Flux")
axes[1].set_xlabel("Time (BTJD)")
axes[1].legend(loc="upper right")
axes[1].set_title("Cleaned & Detrended Input for Period Search")
plt.tight_layout()
plt.show()

## 🔎 Step 5: Period Search & Epoch Verification
We run a Transit Least Squares (TLS) search on the preprocessed light curve to verify the transit epoch and period. We fold the light curve at the discovered period to inspect the transit alignment.

In [ ]:
def run_tls_search(lc, stellar=None):
    time = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)
    flux_err = np.asarray(lc.flux_err.value, dtype=float) if lc.flux_err is not None else None
    
    tls_kwargs = {"period_min": 1.0, "period_max": 15.0}
    if stellar is not None:
        r_star = stellar.get("r_star")
        m_star = stellar.get("m_star")
        r_star_err = stellar.get("r_star_err")
        m_star_err = stellar.get("m_star_err")
        if r_star is not None:
            tls_kwargs["R_star"] = r_star
            tls_kwargs["R_star_min"] = max(0.01, r_star - 3 * (r_star_err or 0.1))
            tls_kwargs["R_star_max"] = r_star + 3 * (r_star_err or 0.1)
        if m_star is not None:
            tls_kwargs["M_star"] = m_star
            tls_kwargs["M_star_min"] = max(0.01, m_star - 3 * (m_star_err or 0.1))
            tls_kwargs["M_star_max"] = m_star + 3 * (m_star_err or 0.1)
            
    model = transitleastsquares(time, flux, flux_err)
    results = model.power(**tls_kwargs)
    return results

print("Running Transit Least Squares (TLS) search...")
tls_result = run_tls_search(lc)

detection = {
    "period": float(tls_result.period),
    "epoch": float(tls_result.T0),
    "duration_hr": float(tls_result.duration) * 24.0,
    "depth": float(tls_result.depth),
    "sde": float(tls_result.SDE),
    "snr": float(tls_result.snr),
}

print("\nTLS Discovered Parameters:")
for key, val in detection.items():
    print(f"  {key}: {val}")

# Plot periodogram power spectrum
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tls_result.periods, tls_result.power, color="steelblue", linewidth=0.8)
ax.axvline(detection["period"], color="red", linestyle="--", label=f"Best P: {detection['period']:.5f} d")
for n in (2, 3):
    ax.axvline(detection["period"] / n, color="orange", linestyle=":", linewidth=0.8, alpha=0.6)
    ax.axvline(detection["period"] * n, color="orange", linestyle=":", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Period (days)")
ax.set_ylabel("TLS Power (SDE)")
ax.set_title(f"Transit Least Squares Spectrum  |  SDE = {detection['sde']:.2f}")
ax.legend()
plt.show()

# Fold and Bin helper functions
def fold_lightcurve(time, flux, flux_err, period, epoch=None):
    t0 = epoch if epoch is not None else time[0]
    phase = ((time - t0) / period) % 1.0
    phase[phase >= 0.5] -= 1.0
    sort_idx = np.argsort(phase)
    return phase[sort_idx], flux[sort_idx], flux_err[sort_idx]

def bin_lightcurve(phase, flux, flux_err, n_bins=100):
    bin_edges = np.linspace(-0.5, 0.5, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    bin_flux = np.full(n_bins, np.nan)
    bin_err = np.full(n_bins, np.nan)
    for i in range(n_bins):
        mask = (phase >= bin_edges[i]) & (phase < bin_edges[i + 1])
        if mask.sum() == 0:
            continue
        w = 1.0 / flux_err[mask]**2 if np.all(flux_err[mask] > 0) else np.ones(mask.sum())
        bin_flux[i] = np.average(flux[mask], weights=w)
        bin_err[i] = 1.0 / np.sqrt(np.sum(w))
    return bin_centers, bin_flux, bin_err

# Plot phase-folded light curve
err_arr = lc.flux_err.value if lc.flux_err is not None else np.ones_like(lc.flux.value) * 1e-3
ph, fl, er = fold_lightcurve(lc.time.value, lc.flux.value, err_arr, detection["period"], detection["epoch"])
bin_c, bin_f, bin_e = bin_lightcurve(ph, fl, er, n_bins=100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(ph, fl, s=1.0, color="steelblue", alpha=0.3, label="Data Points")
valid_bins = np.isfinite(bin_f)
ax.errorbar(bin_c[valid_bins], bin_f[valid_bins], yerr=bin_e[valid_bins], fmt="o", ms=3, color="navy", elinewidth=0.8, capsize=2, label="Binned")
ax.set_xlabel(f"Phase (P = {detection['period']:.5f} d)")
ax.set_ylabel("Normalized Flux")
ax.set_title("Phase-Folded TLS Light Curve")
ax.legend()
plt.show()

## 🌟 Step 6: Stellar Characterization
We query Gaia DR3 for stellar properties using coordinates and then query catalogs (VizieR, Gaia, SIMBAD) to resolve stellar radius ($R_\star$), mass ($M_\star$), and density ($\rho_\star$).

In [ ]:
def characterize_host_star(gaia_params):
    print("Running stellar characterization (multi-catalog)... ")
    from tess_pipeline.catalogs.stellar import characterize_star
    return characterize_star(gaia_params, method="gaia_only")


## 🪐 Step 7: Bayesian Transit Modeling
We compile the transit model using `exoplanet` Keplerian orbit structures and `LimbDarkLightCurve`, with a Gaussian Process (GP) noise kernel (`celerite2` SHO) to model stellar noise. We first optimize to find the Maximum A Posteriori (MAP) solution, and then run NUTS MCMC sampling.

In [ ]:
# Unpack arrays
time_arr = np.asarray(lc.time.value, dtype=float)
flux_arr = np.asarray(lc.flux.value, dtype=float)
flux_err_arr = np.asarray(lc.flux_err.value, dtype=float) if lc.flux_err is not None else np.ones_like(flux_arr) * 1e-3

# Estimates
depth_estimate = float(1.0 - np.percentile(flux_arr, 1))
t0_init = detection["epoch"]
period_val = detection["period"]
rp_init = math.sqrt(depth_estimate)

print(f"Building PyMC model for TIC {target_info['tic_id']}...")
with pm.Model() as model:
    # 1. Stellar Priors
    rho_star = stellar.get("rho_star")
    rho_star_err = stellar.get("rho_star_err")
    if rho_star is not None and rho_star > 0:
        rho_err = rho_star_err if rho_star_err is not None else rho_star * 0.1
        rho_star_var = pm.TruncatedNormal("rho_star", mu=rho_star, sigma=rho_err, lower=0.0, initval=rho_star)
    else:
        rho_star_var = pm.LogNormal("rho_star", mu=np.log(1.4), sigma=1.0, initval=1.4)
        
    # 2. Orbital Priors
    period_var = pm.Normal("period", mu=period_val, sigma=period_val * 0.01, initval=period_val)
    t0_var = pm.Normal("t0", mu=t0_init, sigma=0.1, initval=t0_init)
    
    # 3. Shape Priors
    log_rp = pm.Uniform("log_rp", lower=np.log(0.001), upper=np.log(0.5), initval=np.log(rp_init))
    rp_r_star = pm.Deterministic("rp_r_star", pm.math.exp(log_rp))
    b = pm.Uniform("b", lower=0.0, upper=1.0 + rp_r_star, initval=0.1)
    
    log_t14 = pm.Uniform("log_t14", lower=np.log(0.01), upper=np.log(0.5), initval=np.log(0.1))
    t14 = pm.Deterministic("t14", pm.math.exp(log_t14))
    
    # Kipping (2013) Limb Darkening Parameterization
    q1 = pm.Uniform("q1", lower=0.0, upper=1.0, initval=0.3)
    q2 = pm.Uniform("q2", lower=0.0, upper=1.0, initval=0.3)
    sqrt_q1 = pm.math.sqrt(q1)
    u1 = pm.Deterministic("u1", 2.0 * sqrt_q1 * q2)
    u2 = pm.Deterministic("u2", sqrt_q1 * (1.0 - 2.0 * q2))
    
    # 4. Systematics
    mean_flux = pm.Normal("mean_flux", mu=0.0, sigma=0.01, shape=1, initval=[0.0])
    log_jitter = pm.Normal("log_jitter", mu=-6.0, sigma=2.0, initval=-6.0)
    jitter = pm.Deterministic("jitter", pm.math.exp(log_jitter))
    
    # 5. Keplerian Orbit Setup
    orbit = xo.orbits.KeplerianOrbit(
        period=period_var,
        t0=t0_var,
        b=b,
        duration=t14,
        ror=rp_r_star
    )
    
    # 6. Light Curve Model
    star_lc = xo.LimbDarkLightCurve(u1, u2)
    light_curves = star_lc.get_light_curve(orbit=orbit, r=rp_r_star, t=time_arr)
    transit_model = pm.Deterministic("transit_model", pm.math.sum(light_curves, axis=-1) + mean_flux[0])
    
    # 7. GP Noise Model (celerite2 SHO Kernel)
    log_sigma_gp = pm.Normal("log_sigma_gp", mu=-3.0, sigma=2.0, initval=-3.0)
    log_rho_gp = pm.Normal("log_rho_gp", mu=np.log(10.0), sigma=2.0, initval=np.log(10.0))
    sigma_gp = pm.Deterministic("sigma_gp", pm.math.exp(log_sigma_gp))
    rho_gp = pm.Deterministic("rho_gp", pm.math.exp(log_rho_gp))
    
    term = c2terms.SHOTerm(sigma=sigma_gp, rho=rho_gp, Q=1.0 / np.sqrt(2.0))
    gp = celerite2_pm.GaussianProcess(term, mean=transit_model)
    gp.compute(time_arr, diag=flux_err_arr**2 + pm.math.exp(2.0 * log_jitter), quiet=True)
    gp.marginal("obs", observed=flux_arr)
    
    gp_pred = pm.Deterministic("gp_pred", gp.predict(flux_arr - transit_model))
    flux_model = pm.Deterministic("flux_model", transit_model + gp_pred)
    
    # Dense phase grid model for plotting uncertainty bands
    phase_grid = np.linspace(-0.3, 0.3, 200)
    lc_pred_curves = star_lc.get_light_curve(orbit=orbit, r=rp_r_star, t=t0_var + phase_grid)
    pm.Deterministic("lc_pred", pm.math.sum(lc_pred_curves, axis=-1) + 1.0)
    
    # Optimize to MAP
    print("Optimizing MAP starting parameters...")
    try:
        map_soln = xo.optimize()
    except Exception as e:
        print(f"MAP optimization failed: {e}. Using default starting parameters.")
        map_soln = {}
        
    # Run NUTS Sampling
    # We run a fast setup here (1 chain, 100 draws, 100 tune) for testing.
    print("Sampling parameter posteriors via NUTS MCMC...")
    trace = pm.sample(
        draws=100,
        tune=100,
        chains=1,
        target_accept=0.9,
        initvals=map_soln,
        return_inferencedata=True,
        progressbar=True
    )
    print("Sampling complete!")

## 📊 Step 8: Convergence Diagnostics & Parameter Derivation
We evaluate our MCMC chain convergence properties (ESS, R-hat, and divergences) and compute the derived physical properties of the planet candidate.

In [ ]:
summary = az.summary(trace, round_to=6)

# 1. Convergence stats
rhat_col = "r_hat" if "r_hat" in summary.columns else None
rhat_max = max(summary[rhat_col]) if rhat_col else float("nan")

ess_col = "ess_bulk" if "ess_bulk" in summary.columns else None
ess_min = min(summary[ess_col]) if ess_col else float("nan")

divergences = 0
try:
    divergences = int(trace.sample_stats["diverging"].values.sum())
except:
    pass
    
print("Convergence Diagnostics:")
print(f"  R-hat Max: {rhat_max:.4f}")
print(f"  ESS Min  : {ess_min:.1f}")
print(f"  Diverges : {divergences}")

# 2. Derived Planet Parameters
def get_med_sd(name):
    try:
        row = summary.loc[name]
        return float(row["mean"]), float(row["sd"])
    except:
        return None, None
        
rp_r_star_med, rp_r_star_sd = get_med_sd("rp_r_star")
b_med, b_sd = get_med_sd("b")
t14_med, t14_sd = get_med_sd("t14")
t0_med, t0_sd = get_med_sd("t0")

t14_hr_med = (t14_med * 24.0) if t14_med else None
t14_hr_sd = (t14_sd * 24.0) if t14_sd else None

r_star = stellar.get("r_star")
r_star_err = stellar.get("r_star_err", 0.0) or 0.0

if rp_r_star_med is not None and r_star is not None:
    rp_earth = rp_r_star_med * r_star * 6.957e10 / 6.371e8
    if rp_r_star_sd is not None:
        rp_earth_err = rp_earth * math.sqrt((rp_r_star_sd / rp_r_star_med)**2 + (r_star_err / r_star)**2)
    else:
        rp_earth_err = None
else:
    rp_earth = rp_earth_err = None

m_star = stellar.get("m_star")
try:
    period_med = np.median(trace.posterior["period"].values)
except:
    period_med = period_val
    
if period_med and m_star:
    period_s = period_med * 86400.0
    G = 6.674e-8
    M_SUN = 1.989e33
    R_SUN = 6.957e10
    AU_CGS = 1.496e13
    a_cm = (G * m_star * M_SUN * period_s**2 / (4.0 * math.pi**2))**(1.0 / 3.0)
    a_au = a_cm / AU_CGS
    a_r_star = a_cm / (r_star * R_SUN) if r_star else None
    
    teff = stellar.get("teff")
    t_eq = teff * (0.25 / a_r_star**2)**0.25 if teff and a_r_star else None
else:
    a_au = a_r_star = t_eq = None
    
planet_params = {
    "rp_r_star": rp_r_star_med,
    "rp_r_star_err": rp_r_star_sd,
    "b": b_med,
    "b_err": b_sd,
    "t14_hr": t14_hr_med,
    "t14_hr_err": t14_hr_sd,
    "t0": t0_med,
    "t0_err": t0_sd,
    "rp_earth": rp_earth,
    "rp_earth_err": rp_earth_err,
    "a_au": a_au,
    "a_r_star": a_r_star,
    "t_eq": t_eq,
}
diagnostics = {
    "rhat_max": rhat_max,
    "ess_min": ess_min,
    "divergences": divergences,
}

print("\nDerived Planet Parameters:")
for key, val in planet_params.items():
    if val is not None:
        print(f"  {key:<15}: {val:.5g}")

## 🎨 Step 9: Diagnostic Plots & Report Export
We plot and save the diagnostic figures, compile the final parameter tables, and export our outputs to a structured folder containing an interactive HTML dashboard.

In [ ]:
# 1. Extract Median model curves
flat_samps = trace.posterior.stack(sample=("chain", "draw"))
flux_model_vals = np.median(flat_samps["flux_model"].values, axis=-1) if "flux_model" in flat_samps.data_vars else None
gp_pred_vals = np.median(flat_samps["gp_pred"].values, axis=-1) if "gp_pred" in flat_samps.data_vars else None
transit_model_vals = np.median(flat_samps["transit_model"].values, axis=-1) if "transit_model" in flat_samps.data_vars else None
residuals_vals = flux_arr - flux_model_vals if flux_model_vals is not None else None

model_outputs = {
    "time": time_arr,
    "flux_model": flux_model_vals,
    "gp_model": gp_pred_vals,
    "transit_model": transit_model_vals,
    "residuals": residuals_vals
}

# Create Output Directories
tic_id_val = target_info["tic_id"]
out_dir = Path(f"output/TIC {tic_id_val}")
plots_dir = out_dir / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

# ── Panel Plot 1: raw light curve ───────────────────────────────────────────
fig_raw, ax = plt.subplots(figsize=(10, 4))
ax.plot(stitched_lc.time.value, stitched_lc.flux.value, ".k", alpha=0.3)
ax.set_ylabel("Raw Flux")
ax.set_xlabel("Time (BTJD)")
ax.set_title("Raw Light Curve")
fig_raw.savefig(plots_dir / "raw.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 2: flattened light curve ─────────────────────────────────────
fig_flat, ax = plt.subplots(figsize=(10, 4))
ax.plot(lc.time.value, lc.flux.value, ".b", alpha=0.3)
ax.set_ylabel("Flattened Flux")
ax.set_xlabel("Time (BTJD)")
ax.set_title("Flattened Light Curve")
fig_flat.savefig(plots_dir / "flat.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 3: tls_periodogram ───────────────────────────────────────────
fig_per, ax = plt.subplots(figsize=(12, 4))
ax.plot(tls_result.periods, tls_result.power, color="steelblue", linewidth=0.8)
ax.axvline(detection["period"], color="red", linestyle="--")
ax.set_xlabel("Period (days)")
ax.set_ylabel("TLS Power")
ax.set_title(f"TLS Periodogram")
fig_per.savefig(plots_dir / "tls_periodogram.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 4: phase-folded TLS curve ────────────────────────────────────
fig_phase, ax = plt.subplots(figsize=(10, 5))
ax.scatter(ph, fl, s=1.0, color="steelblue", alpha=0.3, label="Data")
ax.errorbar(bin_c[valid_bins], bin_f[valid_bins], yerr=bin_e[valid_bins], fmt="o", ms=3, color="navy", elinewidth=0.8, capsize=2, label="Binned")
ax.set_xlabel(f"Phase (P = {detection['period']:.5f} d)")
ax.set_ylabel("Normalized Flux")
ax.legend()
fig_phase.savefig(plots_dir / "phase.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 5: Bayesian Fit (3 panels) ───────────────────────────────────
fig_fit, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
axes[0].scatter(time_arr, flux_arr, s=0.5, color="black", alpha=0.3, label="Data", rasterized=True)
if gp_pred_vals is not None:
    trend = flux_model_vals - (transit_model_vals - 1.0) if transit_model_vals is not None else gp_pred_vals + np.median(flux_arr)
    axes[0].plot(time_arr, trend, color="#22c55e", linewidth=1.2, label="GP + Trend")
axes[0].set_ylabel("Relative Flux")
axes[0].legend(loc="upper right")
axes[0].set_title("Bayesian GP Detrending & Transit Fit", fontsize=11, fontweight="bold")

detrended_flux = flux_arr - gp_pred_vals if gp_pred_vals is not None else flux_arr
axes[1].scatter(time_arr, detrended_flux, s=0.5, color="black", alpha=0.3, label="De-trended Data", rasterized=True)
if transit_model_vals is not None:
    axes[1].plot(time_arr, transit_model_vals, color="#2563eb", linewidth=1.2, label="Transit Model")
axes[1].set_ylabel("De-trended Flux")
axes[1].legend(loc="upper right")

axes[2].scatter(time_arr, residuals_vals, s=0.5, color="gray", alpha=0.3, label="Residuals", rasterized=True)
axes[2].axhline(0, color="red", linestyle="--", linewidth=0.8)
axes[2].set_ylabel("Residuals")
axes[2].set_xlabel("Time (BTJD)")
axes[2].legend(loc="upper right")
fig_fit.tight_layout()
fig_fit.savefig(plots_dir / "bayesian_fit.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 6: Corner Plot ───────────────────────────────────────────────
var_names = [v for v in ["period", "rp_r_star", "b", "t14", "u1", "u2", "rho_star"] if v in flat_samps.data_vars]
labels_dict = {"period": "Period [d]", "rp_r_star": "Rₚ/R★", "b": "Impact Parameter b", "t14": "Duration T₁₄ [d]", "u1": "u₁", "u2": "u₂", "rho_star": "Density ρ★ [g/cm³]"}
labels = [labels_dict.get(v, v) for v in var_names]
samples = np.column_stack([flat_samps[v].values for v in var_names])
fig_corner = corner.corner(samples, labels=labels, color="#2563eb", hist_kwargs={"color": "#1d4ed8", "fill": True, "alpha": 0.25}, show_titles=True, title_fmt=".5f")
fig_corner.savefig(plots_dir / "corner.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 7: Trace Plot ────────────────────────────────────────────────
trace_vars = [v for v in ["rp_r_star", "b", "t14", "t0", "q1", "q2", "sigma_gp", "rho_gp"] if v in flat_samps.data_vars]
az.plot_trace(trace, var_names=trace_vars)
fig_trace = plt.gcf()
fig_trace.savefig(plots_dir / "trace.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 8: Posterior Predictive check ────────────────────────────────
fig_pred, ax = plt.subplots(figsize=(12, 5))
ax.scatter(time_arr, flux_arr, s=0.5, color="steelblue", alpha=0.4, label="Data")
if flux_model_vals is not None:
    ax.plot(time_arr, flux_model_vals, color="red", linewidth=1.5, zorder=5, label="Posterior median")
ax.set_xlabel("Time (BTJD)")
ax.set_ylabel("Normalized Flux")
ax.set_title("Posterior Predictive Check")
ax.legend()
fig_pred.savefig(plots_dir / "posterior_predictive.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Panel Plot 9: Residuals vs Time and vs Phase ────────────────────────────
fig_res, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].scatter(time_arr, residuals_vals, s=0.5, color="gray", alpha=0.5)
axes[0].axhline(0, color="red", linewidth=0.8)
axes[0].set_ylabel("Residuals")
axes[0].set_title("Residuals vs Time")

phase_vals, _, _ = fold_lightcurve(lc.time.value, residuals_vals, np.ones_like(residuals_vals)*1e-3, detection["period"], detection["epoch"])
axes[1].scatter(phase_vals, _, s=0.5, color="gray", alpha=0.5)
axes[1].axhline(0, color="red", linewidth=0.8)
axes[1].set_xlabel("Phase")
axes[1].set_ylabel("Residuals")
axes[1].set_title("Residuals vs Phase")
fig_res.savefig(plots_dir / "residuals.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Write CSV summary file ──────────────────────────────────────────────────
rows = []
for key, val in planet_params.items():
    rows.append({"parameter": key, "value": val})
for key, val in stellar.items():
    if isinstance(val, (int, float, str, type(None))):
        rows.append({"parameter": f"stellar_{key}", "value": val})
rows.append({"parameter": "period_d", "value": period_val})

csv_path = out_dir / f"TIC{tic_id_val}_summary.csv"
with csv_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["parameter", "value"])
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved CSV parameters to: {csv_path}")

# ── Write JSON metadata file ────────────────────────────────────────────────
meta_dict = {
    "target": target_info,
    "period": {"value": period_val, "source": "tls"},
    "detection": detection,
    "stellar": stellar,
    "planet": planet_params,
    "diagnostics": diagnostics,
    "metadata": {"author": "walkthrough", "lightcurve_source": "mast"}
}
json_path = out_dir / f"TIC{tic_id_val}_metadata.json"
with json_path.open("w") as f:
    json.dump(meta_dict, f, indent=2, default=str)
print(f"Saved JSON metadata to: {json_path}")

# ── Write NetCDF posterior file ─────────────────────────────────────────────
nc_path = out_dir / f"TIC{tic_id_val}_posterior.nc"
trace.to_netcdf(str(nc_path))
print(f"Saved NetCDF posterior samples to: {nc_path}")

# ── Write FITS lightcurve file ──────────────────────────────────────────────
fits_path = out_dir / f"TIC{tic_id_val}_lightcurve.fits"
lc.to_fits(str(fits_path), overwrite=True)
print(f"Saved preprocessed FITS lightcurve to: {fits_path}")

# ── Generate Interactive HTML Report Dashboard ──────────────────────────────
html_path = out_dir / "report.html"

def get_fmt(val, decimals=4, unit=""):
    if val is None or val == "": return "N/A"
    try:
        return f"{float(val):.{decimals}f}{unit}"
    except:
        return f"{val}{unit}"

html_content = f"""<!DOCTYPE html>
<html lang='en'>
<head>
    <meta charset='UTF-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1.0'>
    <title>TESS Exoplanet Walkthrough Report - TIC {tic_id_val}</title>
    <link href='https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;500;600;700&display=swap' rel='stylesheet'>
    <style>
        :root {{
            --bg-color: #0b0f19;
            --card-bg: rgba(255, 255, 255, 0.03);
            --card-border: rgba(255, 255, 255, 0.08);
            --text-primary: #f3f4f6;
            --text-secondary: #9ca3af;
            --accent: #3b82f6;
            --accent-glow: rgba(59, 130, 246, 0.15);
            --success: #10b981;
            --success-glow: rgba(16, 185, 129, 0.15);
            --warning: #f59e0b;
            --warning-glow: rgba(245, 158, 11, 0.15);
            --danger: #ef4444;
        }}
        * {{
            box-sizing: border-box;
            margin: 0;
            padding: 0;
        }}
        body {{
            font-family: 'Outfit', sans-serif;
            background-color: var(--bg-color);
            color: var(--text-primary);
            line-height: 1.5;
            padding: 2rem;
        }}
        header {{
            margin-bottom: 2rem;
            border-bottom: 1px solid var(--card-border);
            padding-bottom: 1.5rem;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}
        h1 {{
            font-size: 2.2rem;
            font-weight: 700;
            background: linear-gradient(135deg, #60a5fa, #3b82f6);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
        }}
        .subtitle {{
            color: var(--text-secondary);
            font-size: 1rem;
            margin-top: 0.25rem;
        }}
        .grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
            gap: 1.5rem;
            margin-bottom: 2rem;
        }}
        .card {{
            background: var(--card-bg);
            border: 1px solid var(--card-border);
            border-radius: 12px;
            padding: 1.5rem;
            backdrop-filter: blur(10px);
        }}
        .card h2 {{
            font-size: 1.25rem;
            font-weight: 600;
            margin-bottom: 1.25rem;
            border-bottom: 1px solid rgba(255, 255, 255, 0.05);
            padding-bottom: 0.5rem;
            color: #60a5fa;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }}
        .param-row {{
            display: flex;
            justify-content: space-between;
            padding: 0.75rem 0;
            border-bottom: 1px solid rgba(255, 255, 255, 0.02);
            font-size: 0.95rem;
        }}
        .param-row:last-child {{
            border-bottom: none;
        }}
        .param-label {{
            color: var(--text-secondary);
            font-weight: 400;
        }}
        .param-value {{
            font-weight: 500;
            display: flex;
            align-items: center;
            gap: 0.5rem;
        }}
        .badge {{
            font-size: 0.7rem;
            padding: 0.2rem 0.5rem;
            border-radius: 4px;
            text-transform: uppercase;
            font-weight: 600;
            letter-spacing: 0.05em;
            display: inline-block;
        }}
        .badge-derived {{
            background-color: var(--success-glow);
            color: var(--success);
            border: 1px solid rgba(16, 185, 129, 0.3);
        }}
        .badge-literature {{
            background-color: var(--accent-glow);
            color: var(--accent);
            border: 1px solid rgba(59, 130, 246, 0.3);
        }}
        .tabs {{
            display: flex;
            gap: 0.5rem;
            margin-bottom: 1.5rem;
            border-bottom: 1px solid var(--card-border);
            padding-bottom: 0.75rem;
            overflow-x: auto;
        }}
        .tab-btn {{
            background: none;
            border: none;
            color: var(--text-secondary);
            font-family: inherit;
            font-size: 1rem;
            font-weight: 500;
            padding: 0.5rem 1rem;
            cursor: pointer;
            border-radius: 6px;
            transition: all 0.2s;
        }}
        .tab-btn:hover {{
            background: rgba(255, 255, 255, 0.05);
            color: var(--text-primary);
        }}
        .tab-btn.active {{
            background: var(--accent-glow);
            color: #60a5fa;
            border: 1px solid rgba(59, 130, 246, 0.3);
        }}
        .tab-content {{
            display: none;
        }}
        .tab-content.active {{
            display: block;
        }}
        .image-gallery {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(450px, 1fr));
            gap: 1.5rem;
        }}
        .image-card {{
            background: var(--card-bg);
            border: 1px solid var(--card-border);
            border-radius: 12px;
            overflow: hidden;
            display: flex;
            flex-direction: column;
        }}
        .image-card img {{
            width: 100%;
            height: auto;
            border-bottom: 1px solid var(--card-border);
            background: #000;
        }}
        .image-caption {{
            padding: 1rem;
            font-size: 0.9rem;
            color: var(--text-secondary);
            background: rgba(0, 0, 0, 0.2);
        }}
        pre {{
            background: rgba(0, 0, 0, 0.3);
            border: 1px solid var(--card-border);
            padding: 1.5rem;
            border-radius: 8px;
            overflow-x: auto;
            color: #34d399;
            font-family: monospace;
            font-size: 0.9rem;
        }}
    </style>
</head>
<body>
    <header>
        <div>
            <h1>TESS Planet Candidate Dashboard (Standalone)</h1>
            <div class="subtitle">Target: {target_info['name']} | Coordinates: RA={target_info['ra']:.6f}&deg;, Dec={target_info['dec']:.6f}&deg;</div>
        </div>
    </header>

    <main>
        <div class="grid">
            <div class="card">
                <h2>Planet Parameters</h2>
                <div class="param-row">
                    <span class="param-label">Orbital Period (P)</span>
                    <span class="param-value">{get_fmt(period_val, 6, ' d')} <span class="badge badge-derived">Derived (TLS)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Epoch (t₀)</span>
                    <span class="param-value">{get_fmt(planet_params['t0'], 4, ' BTJD')} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Radius Ratio (Rₚ/Rₛ)</span>
                    <span class="param-value">{get_fmt(planet_params['rp_r_star'], 5)} &plusmn; {get_fmt(planet_params['rp_r_star_err'], 5)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Planet Radius (Rₚ)</span>
                    <span class="param-value">{get_fmt(planet_params['rp_earth'], 2, ' R<sub>&oplus;</sub>')} &plusmn; {get_fmt(planet_params['rp_earth_err'], 2)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Transit Duration (T₁₄)</span>
                    <span class="param-value">{get_fmt(planet_params['t14_hr'], 3, ' hr')} &plusmn; {get_fmt(planet_params['t14_hr_err'], 3)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Impact Parameter (b)</span>
                    <span class="param-value">{get_fmt(planet_params['b'], 3)} &plusmn; {get_fmt(planet_params['b_err'], 3)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Semi-major Axis (a)</span>
                    <span class="param-value">{get_fmt(planet_params['a_au'], 4, ' AU')} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Equilibrium Temp (T<sub>eq</sub>)</span>
                    <span class="param-value">{get_fmt(planet_params['t_eq'], 0, ' K')} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
            </div>

            <div class="card">
                <h2>Stellar Parameters</h2>
                <div class="param-row">
                    <span class="param-label">Stellar Radius (Rₛ)</span>
                    <span class="param-value">{get_fmt(stellar['r_star'], 2, ' R<sub>&sub;</sub>')} &plusmn; {get_fmt(stellar['r_star_err'], 2)} <span class="badge badge-derived">Catalog</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Stellar Mass (Mₛ)</span>
                    <span class="param-value">{get_fmt(stellar['m_star'], 2, ' M<sub>&sub;</sub>')} &plusmn; {get_fmt(stellar['m_star_err'], 2)} <span class="badge badge-derived">Catalog</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Effective Temp (T<sub>eff</sub>)</span>
                    <span class="param-value">{get_fmt(stellar['teff'], 1, ' K')} &plusmn; {get_fmt(stellar['teff_err'], 1)} <span class="badge badge-derived">Catalog</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Surface Gravity (log g)</span>
                    <span class="param-value">{get_fmt(stellar['logg'], 2)} &plusmn; {get_fmt(stellar['logg_err'], 2)} <span class="badge badge-derived">Catalog</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Metallicity ([Fe/H])</span>
                    <span class="param-value">{get_fmt(stellar['feh'], 2)} &plusmn; {get_fmt(stellar['feh_err'], 2)} <span class="badge badge-derived">Catalog</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Stellar Density (ρₛ)</span>
                    <span class="param-value">{get_fmt(stellar['rho_star'], 3, ' g/cm&sup3;')} &plusmn; {get_fmt(stellar['rho_star_err'], 3)} <span class="badge badge-derived">Catalog</span></span>
                </div>
            </div>

            <div class="card">
                <h2>MCMC Diagnostics</h2>
                <div class="param-row">
                    <span class="param-label">R-hat Convergence</span>
                    <span class="param-value">{get_fmt(diagnostics['rhat_max'], 4)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Min Effective Sample Size</span>
                    <span class="param-value">{get_fmt(diagnostics['ess_min'], 0)} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">Divergent Transitions</span>
                    <span class="param-value">{diagnostics['divergences']} <span class="badge badge-derived">Derived (MCMC)</span></span>
                </div>
                <div class="param-row">
                    <span class="param-label">TLS SDE / SNR</span>
                    <span class="param-value">{get_fmt(detection['sde'], 2)} / {get_fmt(detection['snr'], 2)} <span class="badge badge-derived">Derived (TLS)</span></span>
                </div>
            </div>
        </div>

        <div class="tabs">
            <button class='tab-btn active' onclick='openTab(event, "tab-lcs")'>Light Curves</button>
            <button class='tab-btn' onclick='openTab(event, "tab-search")'>Period Search</button>
            <button class='tab-btn' onclick='openTab(event, "tab-mcmc")'>MCMC Fit</button>
            <button class='tab-btn' onclick='openTab(event, "tab-json")'>JSON Metadata</button>
        </div>

        <div id='tab-lcs' class='tab-content active'>
            <div class="image-gallery">
                <div class="image-card">
                    <img src="plots/raw.png" alt="Raw Light Curve">
                    <div class="image-caption">Raw stitched light curve before detrending.</div>
                </div>
                <div class="image-card">
                    <img src="plots/flat.png" alt="Flattened Light Curve">
                    <div class="image-caption">Cleaned and flattened light curve ready for analysis.</div>
                </div>
            </div>
        </div>

        <div id='tab-search' class='tab-content'>
            <div class="image-gallery">
                <div class="image-card">
                    <img src="plots/tls_periodogram.png" alt="TLS Periodogram">
                    <div class="image-caption">TLS Periodogram spectrum.</div>
                </div>
                <div class="image-card">
                    <img src="plots/phase.png" alt="Phase-Folded TLS Light Curve">
                    <div class="image-caption">Phase-folded curve folded at the TLS period.</div>
                </div>
            </div>
        </div>

        <div id='tab-mcmc' class='tab-content'>
            <div class="image-gallery">
                <div class="image-card">
                    <img src="plots/bayesian_fit.png" alt="Bayesian GP & Transit Fit">
                    <div class="image-caption">GP noise detrending and MCMC Keplerian transit model fit.</div>
                </div>
                <div class="image-card">
                    <img src="plots/posterior_predictive.png" alt="Posterior Predictive Check">
                    <div class="image-caption">Posterior predictive transit model overlays.</div>
                </div>
                <div class="image-card">
                    <img src="plots/residuals.png" alt="Residuals vs Time & Phase">
                    <div class="image-caption">MCMC transit fit residuals vs time and phase.</div>
                </div>
                <div class="image-card">
                    <img src="plots/corner.png" alt="Posterior Corner Plot">
                    <div class="image-caption">Posterior parameters covariance distribution.</div>
                </div>
                <div class="image-card">
                    <img src="plots/trace.png" alt="Trace Plots">
                    <div class="image-caption">Parameter traces across chain steps.</div>
                </div>
            </div>
        </div>

        <div id='tab-json' class='tab-content'>
            <pre><code>{json.dumps(meta_dict, indent=2, default=str)}</code></pre>
        </div>
    </main>

    <script>
        function openTab(evt, tabId) {{
            const contents = document.getElementsByClassName("tab-content");
            for (let i = 0; i < contents.length; i++) {{
                contents[i].classList.remove("active");
            }}
            const buttons = document.getElementsByClassName("tab-btn");
            for (let i = 0; i < buttons.length; i++) {{
                buttons[i].classList.remove("active");
            }}
            document.getElementById(tabId).classList.add("active");
            evt.currentTarget.classList.add("active");
        }}
    </script>
</body>
</html>"""

with html_path.open("w") as f:
    f.write(html_content)
print(f"Generated HTML report at: {html_path}")
print(f"\nAll standalone walkthrough analysis files generated successfully at: output/TIC {tic_id_val}/")